In [0]:
%run ./00_imports

In [0]:

# Récupération de l'URL des fichiers via l'API data.gouv.fr 
DATASET_SLUG = "resultats-du-controle-sanitaire-de-leau-distribuee-commune-par-commune"

def get_ressources_dataset(slug):
    """Appelle l'API data.gouv.fr avec le slug du dataset."""
    url = f"https://www.data.gouv.fr/api/1/datasets/{slug}/"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()

    print(f"Dataset : {data['title']}")
    print(f"ID : {data['id']}")
    print(f"Nb ressources : {len(data['resources'])}\n")

    ressources = []
    for r in data["resources"]:
        ressources.append({
            "titre"  : r["title"],
            "url"    : r["url"],
            "format" : r.get("format", ""),
            "taille" : r.get("filesize", 0),
        })
    return data["id"], ressources

# Appel API
dataset_id, ressources = get_ressources_dataset(DATASET_SLUG)

# Affichage
for r in ressources:
    print(f"- {r['titre']} | {r['format']} | {r['taille']} octets")
    print(f"  {r['url'][:80]}")
    print()

In [0]:
# Exploration des titres bruts pour comprendre le nommage
# print("Titres de toutes les ressources :\n")
# for r in ressources:
#     print(f"  - {r['titre']}")

In [0]:

# =============================================================
# On filtre les des fichiers nationaux par annee (dis-YYYY.zip)
# ============================================================= 

def filtrer_ressources(ressources, annees):
    """Sélectionne uniquement les fichiers nationaux annuels (dis-YYYY.zip)."""
    selection = []
    for r in ressources:
        for annee in annees:
            pattern = f"dis-{annee}.zip"
            if r["titre"].lower() == pattern:
                selection.append({**r, "annee": annee})
    return selection

ressources_cibles = filtrer_ressources(ressources, ANNEES)

print(f"{len(ressources_cibles)} fichiers sélectionnés :\n")
for r in ressources_cibles:
    print(f"  {r['annee']} | {r['titre']} | {r['taille']} octets")
    print(f"  {r['url'][:80]}")
    print()

In [0]:

# ==========================================================================================================
# On lit uniquement les en-têtes d'un ZIP, pour identifier les fichiers PLV, RESULT et UDI_COM à l'intérieur
# ==========================================================================================================

import zipfile
import io

def lister_contenu_zip(url):
    """Telecharge et liste les fichiers contenus dans un ZIP distant."""
    print(f"Lecture du ZIP : {url[:80]}...")
    response = requests.get(url, timeout=120, stream=True)
    response.raise_for_status()
    
    contenu = io.BytesIO(response.content)
    with zipfile.ZipFile(contenu) as z:
        fichiers = z.namelist()
    return fichiers

# On explore uniquement le ZIP 2024 pour comprendre la structure
url_2024 = [r for r in ressources_cibles if r["annee"] == 2024][0]["url"]
fichiers_zip = lister_contenu_zip(url_2024)

print(f"\n{len(fichiers_zip)} fichiers dans le ZIP 2024 :\n")
for f in fichiers_zip:
    print(f"  - {f}")

In [0]:

# ============================================================================
# On lit les premières lignes de RESULT pour identifier séparateur et colonnes
# ============================================================================

# Chargement uniquement du fichier RESULT du ZIP 2024 
def lire_fichier_zip(url, nom_fichier):
    """Extrait et retourne le contenu d'un fichier specifique depuis un ZIP distant."""
    response = requests.get(url, timeout=120, stream=True)
    response.raise_for_status()
    
    contenu = io.BytesIO(response.content)
    with zipfile.ZipFile(contenu) as z:
        with z.open(nom_fichier) as f:
            lignes = f.read().decode("utf-8", errors="replace")
    return lignes

# Lecture des 3 premières lignes de RESULT 2024
url_2024 = [r for r in ressources_cibles if r["annee"] == 2024][0]["url"]
nom_result = FICHIERS["result"].replace("{annee}", "2024")

contenu_result = lire_fichier_zip(url_2024, nom_result)
lignes = contenu_result.split("\n")[:3]

print(f"Fichier : {nom_result}")
print(f"Separateur detecte : '{lignes[0][20:21]}'")
print(f"\nLigne 1 (en-tetes) :\n{lignes[0][:200]}")
print(f"\nLigne 2 (premiere donnee) :\n{lignes[1][:200]}")

In [0]:

# ===================================================================
# Lecture et ingestion Bronze — fichier RESULT (résultats d'analyses) 
# Années 2023, 2024, 2025
# ===================================================================


# Écriture dans un Volume Unity Catalog puis lecture Spark pour ingestion Bronze
# Le serverless n'autorise pas sparkContext ni /tmp — on utilise les Volumes UC

# Création du schéma et du volume si nécessaire
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.raw_data")

def ingerer_fichier_bronze(url, nom_fichier, annee, table_cible):
    """Écrit le fichier dans un Volume UC puis le lit avec Spark."""
    
    print(f"Téléchargement ZIP {annee}...")
    response = requests.get(url, timeout=300, stream=True)
    response.raise_for_status()
    
    # Extraction depuis le ZIP en mémoire
    contenu_zip = io.BytesIO(response.content)
    with zipfile.ZipFile(contenu_zip) as z:
        with z.open(nom_fichier) as f:
            contenu_txt = f.read().decode("utf-8", errors="replace")
    
    print(f"Fichier extrait : {nom_fichier} ({len(contenu_txt)} caractères)")
    
    # Écriture dans le Volume Unity Catalog
    chemin_volume = f"/Volumes/{CATALOG}/bronze/raw_data/{nom_fichier}"
    with open(chemin_volume, "w", encoding="utf-8") as f:
        f.write(contenu_txt)
    
    print(f"Fichier écrit dans le Volume : {chemin_volume}")
    
    # Lecture avec Spark depuis le Volume
    df = spark.read \
        .option("header", "true") \
        .option("sep", CONFIG["pipeline"]["separateur_csv"]) \
        .option("quote", '"') \
        .option("inferSchema", "false") \
        .csv(chemin_volume)
    
    # Ajout des métadonnées d'ingestion
    df = df \
        .withColumn("_annee", F.lit(annee)) \
        .withColumn("_ingestion_timestamp", F.current_timestamp()) \
        .withColumn("_source_url", F.lit(url))
    
    nb_lignes = df.count()
    print(f"Lignes lues : {nb_lignes}")
    
    # Écriture en Delta Bronze
    df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"_annee = {annee}") \
    .partitionBy("_annee") \
    .saveAsTable(table_cible)
    
    print(f"Table Bronze écrite : {table_cible} | {nb_lignes} lignes | année {annee}")
    return nb_lignes

# Ingestion pour chaque année
total = 0
for ressource in ressources_cibles:
    annee       = ressource["annee"]
    url         = ressource["url"]
    nom_fichier = FICHIERS["result"].replace("{annee}", str(annee))
    nb          = ingerer_fichier_bronze(url, nom_fichier, annee, TABLES["bronze_result"])
    total      += nb
    print()

print(f"Ingestion Bronze terminée : {total} lignes au total")

In [0]:

# ========================================================
# Lecture et ingestion Bronze — fichier PLV (prélèvements)
# Années 2023, 2024, 2025
# ========================================================


# Réutilisation de la fonction ingerer_fichier_bronze avec le fichier PLV
total_plv = 0
for ressource in ressources_cibles:
    annee       = ressource["annee"]
    url         = ressource["url"]
    nom_fichier = FICHIERS["plv"].replace("{annee}", str(annee))
    nb          = ingerer_fichier_bronze(url, nom_fichier, annee, TABLES["bronze_plv"])
    total_plv  += nb
    print()

print(f"Ingestion PLV terminée : {total_plv} lignes au total")

In [0]:

# ============================================================================
# Ingestion des fichiers UDI_COM (liens communes / unités de distribution)
# Années 2023, 2024, 2025
# ============================================================================

total_udi = 0
for ressource in ressources_cibles:
    annee       = ressource["annee"]
    url         = ressource["url"]
    nom_fichier = FICHIERS["udi_com"].replace("{annee}", str(annee))
    nb          = ingerer_fichier_bronze(url, nom_fichier, annee, TABLES["bronze_udi"])
    total_udi  += nb
    print()

print(f"Ingestion UDI_COM terminée : {total_udi} lignes au total")

In [0]:

# ==================================================================================
# On vérifie que les 3 tables sont bien peuplées avant de passer à la couche Silver
# ==================================================================================

for table in [TABLES["bronze_result"], TABLES["bronze_plv"], TABLES["bronze_udi"]]:
    df = spark.table(table)
    print(f"Table : {table}")
    print(f"  Lignes totales : {df.count()}")
    df.groupBy("_annee").count().orderBy("_annee").show()
    print()